# Chess-Coach Agent — LoRA on Kaggle T4

Trains a **Gemma 4 E2B** LoRA adapter on the v1.2 skill-routing SFT corpus, then zips the adapter for download. You serve it locally as q4_0 GGUF on your RTX 4060.

## The real fixes (read this)
Earlier runs failed for three separate reasons, all now resolved:
1. **GPU-1 OOM** — the trainer used `device_map="auto"`, which *sharded* the model across both T4s and overloaded GPU 1. Fixed: it always loads on one GPU (towers on CPU), never shards.
2. **torchao crash** — Kaggle ships torchao 0.10; recent PEFT raises on it inside `get_peft_model`. Fixed: Cell 4 uninstalls torchao.
3. **Slow/"hung" data prep** — assistant-mask tokenization was O(n²) per row. Fixed: O(n) with progress logs.

## Precision: keep 4-bit on a T4
The T4 has only ~14.5GB usable. bf16 E2B (~10GB weights) OOMs at seq 1280. **4-bit QLoRA** (~2.5GB) fits with room and matches the q4_0 GGUF you serve. So `NO_4BIT = False` (Cell 1). QLoRA adapter quality tracks full fine-tuning, so no quality loss for our use.

## Kaggle setup (do this first)
1. **Settings → Accelerator → GPU T4 x2** (single T4 is enough; x2 fine — trainer pins to GPU 0).
2. **Settings → Internet → On** (clone repo + download base model).
3. **Add-ons → Secrets** → add `HF_TOKEN` (HF token, read scope). Accept the Gemma license once: https://huggingface.co/google/gemma-4-E2B-it-qat-q4_0-unquantized
4. Run cells **top to bottom**.

## Corpus
Cell 6 reads the regenerated v1.2 corpus (gzipped on the branch). If a cell reports `rows= 0`, pull the branch again.

In [ ]:
# Cell 1 — config
import os

MODEL = "gemma4_e2b"  # E2B fits ONE T4 in 4-bit. E4B untested on T4 — leave on E2B.
HF_REPO = {
    "gemma4_e4b": "google/gemma-4-E4B-it-qat-q4_0-unquantized",
    "gemma4_e2b": "google/gemma-4-E2B-it-qat-q4_0-unquantized",
}[MODEL]

# Keep this FALSE on a T4. The trainer loads on ONE GPU (towers on CPU) and
# never shards, so the only question is weight size vs the T4's ~14.5GB:
#   NO_4BIT=False -> 4-bit QLoRA, ~2.5GB weights -> FITS with room (use this).
#   NO_4BIT=True  -> bf16, ~10GB weights -> OOMs at seq 1280 on a 14.5GB T4.
# 4-bit also matches the q4_0 GGUF you serve on the 4060.
NO_4BIT = False

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_kaggle"

# training knobs (QLoRA, single-T4-friendly)
MAX_STEPS = 800
RANK = 8
TARGETS = "qv"
GRAD_ACCUM = 8
MAX_SEQ = 1280
print("config:", MODEL, HF_REPO, "no_4bit=", NO_4BIT)


In [ ]:
# Cell 2 — GPU preflight (stop early if no GPU)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — set Accelerator to GPU T4 x2"
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0),
      "| count", torch.cuda.device_count())


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps (Gemma 4 needs a recent transformers)
import subprocess, sys
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=5.8", "accelerate>=1.2", "peft>=0.14", "trl>=0.13",
    "bitsandbytes>=0.45", "datasets", "sentencepiece", "protobuf", "python-chess")
# Kaggle ships torchao 0.10.0; recent PEFT's is_torchao_available() RAISES on
# versions <0.16 instead of returning False, crashing get_peft_model even though
# we never use torchao. Remove it so PEFT skips the torchao dispatcher.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in ("v1_2_train.jsonl", "v1_2_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 7 — train LoRA adapter under runs/<OUTPUT>
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--max-steps", str(MAX_STEPS), "--rank", str(RANK),
       "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM), "--max-seq", str(MAX_SEQ),
       "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")  # bf16 (~10GB) does NOT fit a 14.5GB T4 at seq 1280 -> keep NO_4BIT=False
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})


In [ ]:
# Cell 8 — zip the adapter to /kaggle/working for download
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("adapter files:", os.listdir(run_dir))
out_zip = f"/kaggle/working/{OUTPUT}_adapter"
shutil.make_archive(out_zip, "zip", run_dir)
print("download from the Output panel:", out_zip + ".zip")
